# Spend Analytics: ML development on Cloudera AI

The nightly pipeline runs three CAI jobs: `spend-feature-engineering` → `spend-model-training` →
`spend-model-evaluation`. This notebook runs **the same three functions** (`spend/pipeline.py`) step by
step on today's gold data, and opens up each model in between.

| Step | Job it mirrors | What it does here |
|---|---|---|
| 1 | `spend-feature-engineering` | pins the Iceberg snapshots, reads the features and cohort series |
| 2 | `spend-model-training` | personas (KMeans), categoriser (MiniLM + n-grams), forecast (Chronos-2), signals |
| 3 | `spend-model-evaluation` | the KPI gate, **dry run** |

**Why this matters:** the notebook is not a separate prototype. It imports the same `spend/` package as
the nightly jobs, so there is no "works in the notebook, rewrite it for production" gap.

**Nothing is published.** Stage hand-off goes to `artifacts/demo/`, apart from the nightly
`artifacts/<date>/`. Evaluation runs with `write=False`: no gold table, MLflow run or registry version is
written. Session: Python 3.11, 4 vCPU / 16 GB. Avoid 02:30-03:20 IST, while the nightly DAG runs.

The data is synthetic, with planted archetypes and a planted demo customer, `C00016807`: a Bengaluru
affluent foodie with a Goa trip. Say so up front.

> **Say:** "Same code, same data, same gate as production. The only difference is that I'm not allowed to
> publish."

In [ ]:
import json
import logging
import os
import sys
import time
from datetime import date
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
os.environ.setdefault("HF_HOME", str(ROOT / ".hf_cache"))

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from spend import gate as kpi_gate
from spend.categorise import SeedIndex, assign, load_embedder, normalise
from spend.config import policy
from spend.forecast import load_forecaster
from spend.pipeline import artifacts_dir, evaluate_models, feature_engineering, train_models
from spend.storage import get_storage

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s: %(message)s", force=True)
pd.set_option("display.max_colwidth", 80)

DEMO_ROOT = ROOT / "artifacts" / "demo"
DEMO_CIF = "C00016807"        # planted demo customer: Bengaluru affluent foodie with a Goa trip
POLICY = policy()
storage = get_storage()       # impala on CAI; SPEND_STORAGE_BACKEND=parquet on a laptop
print(f"backend: {storage.name} | hand-off folder: {DEMO_ROOT}")

## Step 1. Feature engineering

**What:** reads the latest snapshot of `gold.spend_features` (one row per customer, 13-week and 4-week
behaviour), the snapshot 4 weeks earlier, the cohort × category weekly series and the categoriser
holdouts, then writes `manifest.json`.

**Why:** reproducibility and auditability. Every Iceberg snapshot ID used (`features_snapshot_id`,
`cohort_snapshot_id`, `map_snapshot_id`) is pinned, so any run can be replayed with
`FOR SYSTEM_VERSION AS OF <id>`. That answers the model-risk question: "show me exactly what data trained
this model on that date." `feature_hash` ties the features to the code version. The relocation signal
needs a "before" view, which is why the 4-weeks-earlier snapshot is read.

**How:** Iceberg time travel on lakehouse tables, with the manifest kept as a run artefact. The second
cell shows the demo customer's feature vector: 13-week spend and transactions, digital and home-city
share, category shares, and 4-week travel and away-from-home share.

> **Say:** "Features live in governed Iceberg tables, not a CSV on someone's laptop. Lineage and time travel
> come from the platform, not extra engineering."

In [ ]:
t = time.time()
manifest = feature_engineering(storage, triggered_by="notebook-demo", root=DEMO_ROOT)
RUN_DATE = date.fromisoformat(manifest["run_date"])
D = artifacts_dir(RUN_DATE, DEMO_ROOT)
print(f"prepared in {time.time() - t:.1f}s -> {D}")

keys = ["run_id", "run_date", "prior_snapshot_date", "features_snapshot_id", "cohort_snapshot_id",
        "map_snapshot_id", "n_customers", "n_series", "cohort_weeks", "map_keys", "uncategorised_share",
        "prev_run_date", "champion_run_id", "feature_hash"]
pd.DataFrame({"value": [str(manifest[k]) for k in keys]}, index=keys)

In [ ]:
feats = pd.read_parquet(D / "features.parquet")
print(f"{len(feats):,} customers x {feats.shape[1]} columns")
cols = ["segment", "home_city", "city_tier", "spend_13w", "txn_13w", "digital_share", "home_share",
        "cat_travel_share", "cat_food_delivery_share", "cat_dining_share", "cat_travel_4w", "away_share_4w"]
feats.set_index("cif").loc[[DEMO_CIF], cols].T

## Step 2. Training

**What:** loads two pretrained models once, then runs the whole `spend-model-training` job, about 45 s
on 4 vCPU / 16 GB:
- **MiniLM** (`all-MiniLM-L6-v2`, about 22M parameters, 384 dimensions): a sentence embedder.
- **Chronos-2:** a time-series foundation model.

**Why:** CPU is enough; this workload needs no GPU, which is an honest sizing point. The models are cached
in `HF_HOME`, so they also work air-gapped or on private cloud when pre-staged.

**How:** `train_models()` produces four outputs: personas, categoriser, forecast and signals.
`stage_durations` gives the time each one took. The sections below open up each output.

In [ ]:
t = time.time()
embedder, forecaster = load_embedder(), load_forecaster()
print(f"{embedder.model_id} | {forecaster.model_id} on {getattr(forecaster, 'device', 'cpu')} "
      f"loaded in {time.time() - t:.1f}s")

t = time.time()
metrics = train_models(RUN_DATE, root=DEMO_ROOT, embedder=embedder, forecaster=forecaster)
print(f"trained in {time.time() - t:.1f}s; stage seconds: {metrics['stage_durations']}")

### 2a. Personas: KMeans on 27 standardised features

**What:** unsupervised segmentation of customers into 5-9 behavioural personas.

**Why:** marketing needs segments based on behaviour, not only the bank's static segment (affluent, mass):
two "affluent" customers can spend very differently. Persona flow also shows migration, such as who moved
from "Tier-2 commuter" to "Tier-1 traveller" since the previous run.

**How:**
- **Why KMeans:** each persona is a centroid profile you can describe in a sentence. It scales to
  millions of rows, and scoring at the endpoint is deterministic and cheap.
- **Choosing k:** every k from 5 to 9 is scored by silhouette (cluster separation, from -1 to 1).
  Silhouette is O(n²), so it runs on a 20,000-row sample. The previous run's k is kept unless another
  beats it by more than 0.01, so the segmentation doesn't flip on noise.
- **Stable IDs:** KMeans labels are arbitrary each run. The Hungarian algorithm matches this run's
  clusters to the previous run's, so "Tier-1 digital foodie" keeps its ID and name, and campaigns that
  key off persona IDs keep working.
- **Quality checks:** silhouette ≥ 0.15 and persona stability (share of customers keeping their persona)
  ≥ 0.80. ARI against the planted archetypes is a synthetic-data sanity check only; say so.

> **Caveat to volunteer:** a silhouette floor of 0.15 is modest. Real spend data rarely clusters cleanly;
> the aim is a stable, usable segmentation, not textbook clusters.

In [ ]:
sil = pd.Series(metrics["silhouette_by_k"]).rename_axis("k").reset_index(name="silhouette")
fig = px.bar(sil, x="k", y="silhouette", title=f"Silhouette by k (chosen: k = {metrics['k']})")
fig.add_hline(y=POLICY["gate"]["min_silhouette"], line_dash="dash", annotation_text="gate floor")
fig.show()

print(f"stability vs previous run: {metrics['persona_stability']} | "
      f"ARI vs the planted archetypes: {metrics['ari_hidden']}")
profile = pd.read_parquet(D / "persona_profile.parquet")
profile[["persona_id", "persona_name", "n_customers", "customer_share", "avg_spend_13w", "top_category",
         "top_channel", "digital_share", "home_share"]].sort_values("n_customers", ascending=False).round(3)

In [ ]:
cp = pd.read_parquet(D / "customer_persona.parquet")
display(cp[cp["cif"] == DEMO_CIF][["cif", "home_city", "segment", "persona_name", "prev_persona_id",
                                   "top_category", "spend_13w", "distance"]])

flow_path = D / "persona_flow.parquet"
if flow_path.exists():
    flow = pd.read_parquet(flow_path)
    names = sorted(set(flow["from_persona_name"]) | set(flow["to_persona_name"]))
    idx = {n: i for i, n in enumerate(names)}
    fig = go.Figure(go.Sankey(
        node=dict(label=[f"{n} (prev)" for n in names] + names, pad=12),
        link=dict(source=flow["from_persona_name"].map(idx), target=flow["to_persona_name"].map(idx) + len(names),
                  value=flow["n_customers"])))
    fig.update_layout(title=f"Persona flow since {flow['prev_run_date'].iloc[0]}", height=550)
    fig.show()

### 2b. Categoriser: MiniLM embeddings + character n-grams

**What:** gives every merchant a spend category (TRAVEL, FOOD_DELIVERY, GROCERY and so on), including
UPI payees.

**Why:** the India-specific hook. A UPI payment carries only a free-text payee name such as
"RAMESH KIRANA STORE", with no MCC. Without categorisation, a large share of spend is invisible to
analytics.

**How:**
- **A cascade, MCC → SEED → EMBEDDING → FALLBACK:** a card MCC wins; then an exact match in the curated
  seed list; otherwise a kNN search over the seed merchants' embeddings.
- **Why retrieval, not a trained classifier:** there aren't enough labels for the long tail of payees.
  Retrieval generalises to unseen names, and extending it means adding seed merchants, with no
  retraining. Every prediction reports `nearest_seed` and `similarity`, so it is explainable.
- **Hybrid similarity** = (1 − w) × MiniLM cosine + w × character n-gram TF-IDF cosine (`char_wb`, 2-4
  grams, w = 0.5). MiniLM brings meaning; the n-grams bring spelling: `SWIGGY*BLR 5521`, truncations and
  typos that MiniLM's word pieces lose. The 5 nearest seeds vote, weighted by similarity, plus a
  taxonomy prior on similarity to each category description. Below 0.45 the nearest seed is not
  trusted: the category descriptions decide, then OTHER.
- **Be open about it:** MiniLM alone missed the 0.85 target; that is why the hybrid exists.
- **Measured two ways:** 20% of seed merchants held out of the index (names it never saw), and up to
  5,000 card merchants with their MCC hidden, categorised from the name alone and checked against the
  real MCC. Both must be ≥ 0.85.

**Live moment:** in the second cell, type payee strings such as "SHARMA MEDICOS", "PAYTM*BESCOM BILL" or
one the audience suggests, and show category, method and nearest seed. Show the misses on purpose: it
builds credibility. `TANISHQ JEWELLERS` falls back to OTHER (a brand with no type word, and no jewellery
category in the taxonomy); that is the honest miss.

In [ ]:
seed_h, mcc_h = pd.read_parquet(D / "seed_holdout.parquet"), pd.read_parquet(D / "mcc_holdout.parquet")
display(pd.DataFrame({"accuracy": [metrics["category_accuracy_seed"], metrics["category_accuracy_mcc"]],
                      "n": [metrics["n_seed_holdout"], metrics["n_mcc_holdout"]],
                      "gate floor": POLICY["gate"]["min_category_accuracy"]},
                     index=["seed holdout", "hidden MCC"]).round(3))

by_cat = (mcc_h.assign(correct=mcc_h["category"] == mcc_h["truth"]).groupby("truth")["correct"]
          .agg(["mean", "size"]).sort_values("mean").reset_index())
px.bar(by_cat, x="mean", y="truth", orientation="h", hover_data=["size"], range_x=[0, 1],
       title="Hidden-MCC accuracy by true category").show()

print("sample misses:")
mcc_h[mcc_h["category"] != mcc_h["truth"]].head(8)[["merchant_key", "truth", "category", "method",
                                                     "similarity", "nearest_seed"]]

In [ ]:
seeds, taxonomy = pd.read_parquet(D / "merchant_seed.parquet"), pd.read_parquet(D / "category_taxonomy.parquet")
index = SeedIndex.build(seeds, taxonomy, embedder)

texts = ["SWIGGY*BLR 5521", "ola cabs pvt", "SHARMA MEDICOS", "RAMESH KIRANA STORE", "PAYTM*BESCOM BILL",
         "TANISHQ JEWELLERS"]   # try your own
live = assign([normalise(x) for x in texts], index, embedder, POLICY["categorise"])
live.insert(0, "payee text", texts)
live.round(3)

### 2c. Forecast: Chronos-2 vs seasonal naive

**What:** a 12-week-ahead weekly spend forecast for 224 series (cohort = segment × city tier, ×
category), with P10/P50/P90 bands.

**Why:** planning campaign budgets, liquidity for card spend and festive-season offers. This is
cohort-level; the endpoint forecasts per customer for their top 3 categories.

**How:**
- **Zero-shot:** Chronos-2 is a pretrained time-series foundation model, with no per-series training.
  Retraining 224 ARIMA or Prophet models every week is a maintenance burden; one pretrained model covers
  them all.
- **Calendar covariates:** festival, salary, month-end, holiday-count and school-fee weeks, which drive
  Indian spend heavily.
- **Quantiles natively:** an uncertainty band, not one number.
- **Backtest:** 4 rolling cut-offs, 4 weeks apart; each hides the next 12 weeks, forecasts them and
  compares with the actuals. The metric is WAPE (weighted absolute percentage error), and the baseline is
  seasonal naive: same week last year × the recent trend.
- **Gate:** WAPE ratio (Chronos-2 / naive) ≤ 0.90, so the model must beat the baseline by at least 10%;
  P10-P90 coverage in [0.60, 0.95], because too narrow is overconfident and too wide is useless.

> **Say:** "The naive baseline is the honest test. If a foundation model can't beat 'same as last Diwali,
> adjusted for growth', it shouldn't ship."

In [ ]:
bt = pd.read_parquet(D / "spend_backtest.parquet")
err = bt.assign(model=(bt["actual"] - bt["p50"]).abs(), naive=(bt["actual"] - bt["naive"]).abs())
w = err.groupby("category")[["model", "naive", "actual"]].sum()
w = pd.DataFrame({"Chronos-2": w["model"] / w["actual"], "seasonal naive": w["naive"] / w["actual"]})
px.bar(w.reset_index().melt(id_vars="category", var_name="model", value_name="WAPE"), x="category", y="WAPE",
       color="model", barmode="group", title=f"Backtest WAPE by category: ratio {metrics['wape_ratio']:.3f} "
       f"(gate <= {POLICY['gate']['max_wape_vs_naive']}), P10-P90 coverage {metrics['coverage']:.3f}").show()

In [ ]:
demo = feats.set_index("cif").loc[DEMO_CIF]
SERIES = f"{demo['cohort']}|TRAVEL"          # the demo customer's cohort; try another category
hist = pd.read_parquet(D / "cohort_weekly.parquet")
hist = hist[(hist["cohort"] + "|" + hist["category"]) == SERIES].sort_values("week_end")
fc_ = pd.read_parquet(D / "spend_forecast.parquet").query("series_id == @SERIES").sort_values("week_end")

fig = go.Figure()
fig.add_scatter(x=hist["week_end"], y=hist["amount_inr"], name="actual", line=dict(color="black"))
fig.add_scatter(x=fc_["week_end"], y=fc_["p90"], line=dict(width=0), showlegend=False)
fig.add_scatter(x=fc_["week_end"], y=fc_["p10"], fill="tonexty", line=dict(width=0), name="P10-P90",
                fillcolor="rgba(31,119,180,0.25)")
fig.add_scatter(x=fc_["week_end"], y=fc_["p50"], name="Chronos-2 P50", line=dict(color="#1f77b4"))
fig.add_scatter(x=fc_["week_end"], y=fc_["naive"], name="seasonal naive", line=dict(dash="dot", color="gray"))
fig.update_layout(title=f"{SERIES}: 12-week forecast (INR / week)", height=450)
fig.show()

### 2d. Signals: rules, not ML

**What:** next-best-action triggers defined in `config/policy.yaml`:

| Signal | Rule | Action |
|---|---|---|
| TRAVEL_SURGE | 4-week travel ≥ 3× the customer's norm and ≥ ₹5,000 | Travel card / forex offer |
| FESTIVE_ELECTRONICS | ≥ ₹50,000 electronics in festival weeks | EMI conversion offer |
| RELOCATION | ≥ 70% of spend away from home in 3+ of 4 weeks, after ≤ 30% before | Offer home-branch change |

**Why:** not everything should be a model. Marketing analytics owns these thresholds. Changing one
re-lists customers on the next run, with no retraining, data-science ticket or model validation cycle.

**How:** the rules run on today's features and are joined with the persona for context. The demo
customer triggers TRAVEL_SURGE because of the Goa trip.

> **Say:** "ML where it earns its keep, rules where the business needs control. Both governed in the same
> pipeline."

In [ ]:
sig = pd.read_parquet(D / "spend_signals.parquet")
display(sig.groupby(["signal_id", "action"]).size().rename("customers").reset_index())
sig[sig["cif"] == DEMO_CIF][["cif", "signal_id", "metric", "metric_value", "threshold", "detail", "persona_name"]]

## Step 3. Evaluation: the KPI gate (dry run)

**What:** an automated go/no-go check on every candidate run before anything is published.

**Why:** the MLOps story: a bad model can't reach production silently. On FAIL, yesterday's personas,
forecast and signals stay live and the DAG goes red, so business users never see a degraded result.

**How:** two kinds of checks.

| Absolute KPI (`config/policy.yaml`) | Threshold |
|---|---|
| Category accuracy, both holdouts | ≥ 0.85 |
| WAPE ratio vs naive | ≤ 0.90 |
| P10-P90 coverage | 0.60-0.95 |
| Silhouette | ≥ 0.15 |
| Persona stability | ≥ 0.80 (skipped on the first run) |

**Non-regression against the champion** (the published run this candidate would replace): accuracy may
not drop by more than 0.01, the WAPE ratio may not rise by more than 0.03, and `feature_hash` must match
the code, so features and code stay in sync. A KPI that couldn't be measured fails its check rather than
passing silently. `write=False`: nothing is written.

In [ ]:
result = evaluate_models(storage, RUN_DATE, write=False, tracker=None, registrar=None, root=DEMO_ROOT)
print("GATE", "PASSED: would publish" if result["gate_passed"] else "FAILED: would keep yesterday's run")
pd.DataFrame([(line[:4], line[5:]) for line in result["gate_detail"]], columns=["result", "check"])

**What a rejection looks like.** The same metrics against a stricter gate (accuracy ≥ 0.95, WAPE ratio
≤ 0.70), built in memory only; `config/policy.yaml` is not touched. Run it: people trust a gate more once
they've seen it say no.

In [ ]:
strict = {**POLICY["gate"], "min_category_accuracy": 0.95, "max_wape_vs_naive": 0.70}
passed, lines = kpi_gate.evaluate({**metrics, "feature_hash": manifest["feature_hash"]}, strict,
                                  code_hash=manifest["feature_hash"], champion=manifest.get("champion_kpis"))
print("GATE", "PASSED" if passed else "REJECTED the candidate")
pd.DataFrame([(line[:4], line[5:]) for line in lines], columns=["result", "check"])

## In production: every night, the same code

**What:** what the nightly DAG actually published, read back from `gold.spend_model_run`: the last 15 run
dates and their KPI trend. Then MLflow and the live endpoint.

**Why:** it closes the loop from notebook to operating system. Show these CAI tabs:
- **Experiments › spend-analytics:** one MLflow run per night, with the KPIs, the gate detail and the
  persona model.
- **Registered Models › spend-persona:** a champion version for every run that passed the gate.
- **Model Deployments › spend-scorer:** a REST endpoint serving the latest published run.

In [ ]:
runs = storage.read("spend_model_run")
runs = runs.sort_values(["run_date", "run_ts"]).drop_duplicates("run_date", keep="last").tail(15)
kpis = ["category_accuracy_seed", "category_accuracy_mcc", "wape_ratio", "coverage", "silhouette",
        "persona_stability"]
display(runs[["run_date", "run_id", "triggered_by", "gate_passed", "k", *kpis]].round(3))
px.line(runs.melt(id_vars="run_date", value_vars=kpis, var_name="kpi"), x="run_date", y="value", color="kpi",
        markers=True, title="KPIs of the published runs").show()

In [ ]:
try:
    import mlflow
    ml = mlflow.search_runs(experiment_names=["spend-analytics"], order_by=["start_time DESC"], max_results=10)
    cols = [c for c in ml.columns if c in ("run_id", "start_time", "status") or c.startswith("metrics.")][:10]
    display(ml[cols])
except Exception as e:
    print(f"MLflow not available here ({type(e).__name__}): open Experiments > spend-analytics in CAI")

### The deployed model: `spend-scorer`

The same artefacts behind a REST endpoint (needs `SPEND_ENDPOINT_URL`, `_ACCESS_KEY` and `_API_KEY` in
the project environment). The cell calls it twice:
- **`profile`** for the demo customer: persona, 13-week spend by category, channel and city tier, home
  vs away with the top away cities, and his signals. (The 12-week forecast is a separate `forecast`
  action, also on the Test tab.)
- **`what_if`** with `{"TRAVEL": 0.0}`: "what would he look like without the Goa trip?" The scorer
  re-assigns the persona on the shifted behaviour in well under a second: he moves to "Tier-1 digital
  foodie".

> **Say:** "Same artefacts, three consumers: gold tables for BI, MLflow for model risk, and an API for the
> app or campaign engine."

In [ ]:
from spend.client import call_endpoint, endpoint_configured

if endpoint_configured():
    for req in ({"action": "profile", "cif": DEMO_CIF},
                {"action": "what_if", "cif": DEMO_CIF, "shift": {"TRAVEL": 0.0}}):
        t = time.time()
        resp = call_endpoint(req)
        print(f"# {req['action']}: {time.time() - t:.2f}s")
        print(json.dumps(resp, indent=1, default=str)[:1500])
else:
    print("SPEND_ENDPOINT_URL not set: use the Test tab of spend-scorer instead")

## Likely questions

- **"Why not an LLM for categorisation?"** Volume, cost and determinism. Each night the categorise job
  embeds every new merchant key on CPU, with explainable output (nearest seed, similarity). MiniLM is
  about 22M parameters; an LLM call per merchant isn't justified. The unresolved FALLBACK cases are where
  an LLM could be considered.
- **"Is Chronos-2 fine-tuned?"** No. It is zero-shot with covariates; the backtest against the naive
  baseline is the justification.
- **"What if the gate fails three nights in a row?"** Yesterday's outputs stay live but go stale, and
  each failed night turns the DAG red. Alerting on consecutive failures is **not built yet**; say it is
  the next step rather than claiming it.
- **"Is this synthetic data?"** Yes: the ARI against planted archetypes and the planted demo customer
  give it away. Be upfront.
- **Timing:** don't run this between 02:30 and 03:20 IST, while the nightly DAG runs, and don't push to
  `main` within 30 minutes of the demo (the GitHub chain resets the project code).